# Trabalho de LPA - G13

Cristiano Jose da Silva  
Francisco de Assis de Lima Filho  
José Prado Leitão  
Marlon Mendonça Pedersoli de Oliveira  
Pedro Belle Magalhães de Castro  
Renato Martins

---
### 1 - Análise Exploratória de dados

---
A partir da base de dados precos_carros_brasil.csv, execute as seguintes tarefas:
1. Carregue a base
2. Verifique se há valores faltantes nos dados. Caso haja, escolha uma tratativa para resolver o problema de valores faltantes
3. Verifique se há dados duplicados nos dados
4. Crie duas categorias, para separar colunas numéricas e categóricas. Imprima o resumo de informações das variáveis numéricas e categóricas (estatística descritiva dos dados)
5. Imprima a contagem de valores por modelo (model) e marca do carro (brand)
6. Dê um breve explicação (máximo de quatro linhas) sobre os principais resultados encontrados na Análise Exploratória dos dados

---

In [1]:
%matplotlib inline
# Carregar bibliotecas
import pandas as pd

# Biblioteca Seaborn - Criação de gráficos
import seaborn as sns

# Biblioteca Matplotlib - Criação de gráficos
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
# Define o formato de saída para SVG
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# Biblioteca matemática
import numpy as np

# OPCIONAL - Biblioteca para ignorar mensagens de warning (aviso) ao rodar uma célula de código
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Bibliotecas de machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.inspection import permutation_importance

# Métricas de avaliação dos modelos
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Para medir o tempo do processo
import time

---
1. Carregue a base de dados precos_carros_brasil.csv

In [ ]:
# # Ler o arquivo .csv e dar um resumo
# df = pd.read_csv("precos_carros_brasil.csv")
# print(f'\nDimensões do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n')

# lst_colunas = df.columns
# print("Lista das colunas e tipos:\n",df.dtypes)

In [3]:
# Se for rodar esse notebook no Google colab, coloque o arquivo csv no seu Google Drive
# usando o seguinte caminho:
# G:\Meu Drive\Colab_projects\projetos_ml\data
# "G:\Meu Drive" vai depender da instalação em cada máquina

## Vamos experimentar rodar o projeto ou via local ou pelo Google colab
file_csv = "precos_carros_brasil.csv"

def running_in_colab():
    try:
        import google.colab # type: ignore
        return True
    except ImportError:
        return False

if running_in_colab():
    from google.colab import drive # type: ignore
    drive.mount('/content/drive', force_remount=True)

    BASE_PATH = "/content/drive/MyDrive/Colab_projects/projetos_ml"
    DATA_PATH = f"{BASE_PATH}/data"

    df = pd.read_csv(f"{DATA_PATH}/{file_csv}")
else:
    df = pd.read_csv(file_csv) # roda com a versão local

In [4]:
print(f'\nDimensões do dataset: {df.shape[0]:,} linhas x {df.shape[1]} colunas\n')

lst_colunas = df.columns
print("Lista das colunas e tipos:\n",df.dtypes)


Dimensões do dataset: 267,542 linhas x 11 colunas

Lista das colunas e tipos:
 year_of_reference     float64
month_of_reference        str
fipe_code                 str
authentication            str
brand                     str
model                     str
fuel                      str
gear                      str
engine_size               str
year_model            float64
avg_price_brl         float64
dtype: object


---
2. Verifique se há valores faltantes nos dados. Caso haja, escolha uma tratativa para
resolver o problema de valores faltantes 

In [ ]:
print("=== Valores faltantes por coluna ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Valores Faltantes': missing, 'Percentual (%)': missing_pct})
display(missing_df[missing_df['Valores Faltantes'] > 0])

In [ ]:
# Tratativa: Remoção das linhas com valores em branco
print(f'\nLinhas antes da limpeza: {len(df)}')
df = df.dropna(how='all')
print(f'Linhas depois da limpeza: {len(df)}')

In [ ]:
# Verificação após tratamento
print(f'\nValores faltantes restantes: {df.isnull().sum().sum()}')

---
3. Verifique se há dados duplicados nos dados

In [ ]:
duplicadas = df.duplicated().sum()
print(f'=== Verificação de Dados Duplicados ===')
print(f'Total de linhas duplicadas: {duplicadas:,}')

In [ ]:
if duplicadas > 0:
  print("\nExemplo de linhas duplicadas:")
  print(df[df.duplicated(keep=False)].head(6))
  df = df.drop_duplicates()
  print(f'\nLinhas após remoção de duplicatas: {len(df):,}')
else:
  print('\nNenhuma linha duplicata encontrada. Nenhuma ação necessária.')

---
4. Crie duas categorias, para separar colunas numéricas e categóricas. Imprima o resumo

In [ ]:
# Criando categorias para separar colunas numéricas e categóricas: facilita a AED
# compatibilizado para funcionar no Pandas 2.2 e 3.x

#colunas_numericas = [col for col in df.columns if df[col].dtype != 'str']
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
#colunas_categoricas = [col for col in df.columns if df[col].dtype == 'str']
colunas_categoricas = df.select_dtypes(exclude="number").columns.tolist()

print("Lista de colunas numéricas ", colunas_numericas)
print("\nLista de colunas categóricas ", colunas_categoricas)

In [ ]:
print('\n=== Estatística Descritiva — Variáveis Numéricas ===')
print(df[colunas_numericas].describe().round(2))

print('\n=== Estatística Descritiva — Variáveis Categóricas ===')
print(df[colunas_categoricas].describe())

---
5. Imprima a contagem de valores por modelo (model) e marca do carro (brand)

In [ ]:
print('=== Contagem de registros por Marca (brand) ===')
contagem_marca = df['brand'].value_counts().reset_index()
contagem_marca.columns = ['Marca', 'Quantidade']
print(contagem_marca)



In [ ]:
print('\n=== Contagem de registros por Modelo (model) — Top 20 ===')
contagem_modelo = df['model'].value_counts().reset_index()
contagem_modelo.columns = ['Modelo', 'Quantidade']
print(contagem_modelo.head(20))



In [ ]:
print(f'\nTotal de modelos únicos: {df["model"].nunique():,}')

In [ ]:
print(f'\nTotal de marcas: {df["brand"].nunique():,}')

---
6. Dê um breve explicação (máximo de quatro linhas) sobre os principais resultados
encontrados na Análise Exploratória dos dados

O dataset contém **267.542 registros** de preços FIPE de veículos no Brasil entre 2021 e 2023, abrangendo **6 marcas** e mais de **500 modelos** distintos. Foram identificadas **65.245 linhas completamente nulas**, removidas por não conterem informação útil, sem impacto nos dados válidos. As variáveis numéricas revelam que o preço médio dos veículos é de aproximadamente **R$ 60.000**, com alta dispersão (desvio padrão elevado), indicando uma base com veículos de perfis variados, desde populares até de alto padrão. A marca com maior volume de registros é a **GM - Chevrolet**, refletindo sua ampla gama de modelos ao longo dos anos analisados.


---
### 2 - Visualização dos dados

---
A partir da base de dados precos_carros_brasil.csv, execute as seguintes tarefas:
1. Gere um gráfico da distribuição da quantidade de carros por marca
2. Gere um gráfico da distribuição da quantidade de carros por tipo de engrenagem do carro
3. Gere um gráfico da evolução da média de preço dos carros ao longo dos meses de 2022 (variável de tempo no eixo X)
4. Gere um gráfico da distribuição da média de preço dos carros por marca e tipo de engrenagem
5. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados gerados no
item 4
6. Gere um gráfico da distribuição da média de preço dos carros por marca e tipo de
combustível
7. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados gerados no
item 6

---
1. Gere um gráfico da distribuição da quantidade de carros por marca

In [ ]:
# Paleta de cores consistente
PALETTE = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800', '#00BCD4']

contagem_marca = df['brand'].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(contagem_marca.index, contagem_marca.values, color=PALETTE, edgecolor='white', linewidth=0.8)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
            f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Distribuição da Quantidade de Carros por Marca', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Marca', fontsize=12)
ax.set_ylabel('Quantidade de Registros', fontsize=12)
ax.tick_params(axis='x', rotation=15)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
2. Gere um gráfico da distribuição da quantidade de carros por tipo de engrenagem do carro

In [ ]:
contagem_gear = df['gear'].value_counts()
labels_pt = {'manual': 'Manual', 'automatic': 'Automático'}
labels = [labels_pt.get(x, x) for x in contagem_gear.index]

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

# Gráfico de barras
bars = axes[0].bar(labels, contagem_gear.values, color=PALETTE, edgecolor='white', width=0.5)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Quantidade por Tipo de Câmbio', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade de Registros', fontsize=11)
#axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de pizza
axes[1].pie(contagem_gear.values, labels=labels, autopct='%1.1f%%',
            colors=PALETTE, startangle=90,
            textprops={'fontsize': 12}, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção por Tipo de Câmbio', fontsize=13, fontweight='bold')

fig.suptitle('Distribuição da Quantidade de Carros por Tipo de Câmbio', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
3. Gere um gráfico da evolução da média de preço dos carros ao longo dos meses de 2022 (variável de tempo no eixo X)

In [ ]:
# Obter os dados dos meses de 2022 e ordernar pelo nome do mês
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

# Label para o gráfico
month_pt = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

df_2022 = df[df['year_of_reference'] == 2022].copy()
df_2022['month_of_reference'] = pd.Categorical(df_2022['month_of_reference'], categories=month_order, ordered=True)
media_mensal = df_2022.groupby('month_of_reference', observed=True)['avg_price_brl'].mean().round(0)



In [ ]:
media_mensal

In [ ]:
# Plotando o gráfico
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(len(media_mensal)), media_mensal.values, marker='o', color=PALETTE[0],
        linewidth=2.5, markersize=8, markerfacecolor='white', markeredgewidth=2.5)
ax.fill_between(range(len(media_mensal)), media_mensal.values, alpha=0.1, color=PALETTE)

for i, v in enumerate(media_mensal.values):
    ax.annotate(f'R$ {v:,.0f}', (i, v), textcoords='offset points', xytext=(0, 12),
                ha='center', fontsize=8.5, fontweight='bold', color=PALETTE[0])

ax.set_xticks(range(len(media_mensal)))
ax.set_xticklabels(month_pt, fontsize=11)
ax.set_title('Evolução da Média de Preço dos Carros em 2022', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Mês', fontsize=12)
ax.set_ylabel('Preço Médio (R$)', fontsize=12)
#ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
4. Gere um gráfico da distribuição da média de preço dos carros por marca e tipo de engrenagem

In [ ]:
# Agrupado os dados
media_marca_gear = df.groupby(['brand', 'gear'])['avg_price_brl'].mean().unstack()
media_marca_gear.columns = ['Automático', 'Manual']
media_marca_gear = media_marca_gear.sort_values('Automático', ascending=False)

x = range(len(media_marca_gear))

width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar([i - width/2 for i in x], media_marca_gear['Automático'],
               width, label='Automático', color=PALETTE[0], edgecolor='white')
bars2 = ax.bar([i + width/2 for i in x], media_marca_gear['Manual'],
               width, label='Manual', color=PALETTE[1], edgecolor='white')

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
            f'R$ {bar.get_height()/1000:.0f}k', ha='center', va='bottom', fontsize=8.5, fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(media_marca_gear.index, rotation=15, ha='right', fontsize=10)
ax.set_title('Média de Preço dos Carros por Marca e Tipo de Câmbio', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Marca', fontsize=12)
ax.set_ylabel('Preço Médio (R$)', fontsize=12)

ax.legend(title='Câmbio', fontsize=11, title_fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
5. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados gerados no
item 4

Em todas as marcas analisadas, os veículos com câmbio **automático** apresentam preço médio consistentemente superior aos de câmbio **manual**, o que reflete o maior custo tecnológico e o perfil premium desse tipo de transmissão. A **Nissan** se destaca com a maior diferença entre os dois câmbios, sugerindo um portfólio automático voltado a segmentos de maior valor agregado. A **Fiat** e a **VW - VolksWagen** apresentam os menores preços médios em ambas as categorias, consolidando seu posicionamento no segmento de veículos populares. Já a **Ford** aparece com médias mais elevadas, possivelmente influenciada por modelos como caminhonetes e SUVs em sua composição de portfólio.

---
6. Gere um gráfico da distribuição da média de preço dos carros por marca e tipo de
combustível

In [ ]:
# Agrupa por marca e tipo de combustivel e obtem o preço médio
media_marca_fuel = df.groupby(['brand', 'fuel'])['avg_price_brl'].mean().unstack()

# Ordena 
media_marca_fuel = media_marca_fuel.sort_values('Diesel' if 'Diesel' in media_marca_fuel.columns else media_marca_fuel.columns[0], ascending=False)

# Cores para exibição
fuel_colors = {'Alcohol': PALETTE[0], 'Diesel': PALETTE[1], 'Gasoline': PALETTE[2]}

# Monta o gráfico
x = range(len(media_marca_fuel))
n_fuels = len(media_marca_fuel.columns)
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
for i, fuel in enumerate(media_marca_fuel.columns):
    offset = (i - n_fuels / 2 + 0.5) * width
    bars = ax.bar([xi + offset for xi in x], media_marca_fuel[fuel],
                  width, label=fuel, color=fuel_colors.get(fuel, 'black'), edgecolor='white')
    for bar in bars:
        height = bar.get_height()
        if not (height != height):  # ignora NaN
            ax.text(bar.get_x() + bar.get_width() / 2, height + 200,
                    f'R$ {height/1000:.0f}k', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(media_marca_fuel.index, rotation=15, ha='right', fontsize=10)
ax.set_title('Média de Preço dos Carros por Marca e Tipo de Combustível', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Marca', fontsize=12)
ax.set_ylabel('Preço Médio (R$)', fontsize=12)

ax.legend(title='Combustível', fontsize=11, title_fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
7. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados gerados no
item 6

Os veículos **a diesel** apresentam os maiores preços médios em todas as marcas que oferecem essa opção, especialmente na **GM - Chevrolet** e na **Ford**, que concentram caminhonetes e utilitários nesse segmento. Os carros **a gasolina** ocupam uma posição intermediária, enquanto os **a álcool** tendem a ter os menores preços médios, por serem associados a modelos mais antigos ou populares com motorização flex. A **Nissan** e a **Renault** não apresentam versões a diesel no dataset, o que reforça seu foco em veículos de passeio e urbanos. De forma geral, o tipo de combustível é um bom indicador indireto do segmento e porte do veículo.

---
### 3 Aplicação de modelos de machine learning para prever o preço médio dos carros

---
A partir da base de dados precos_carros_brasil.csv, execute as seguintes tarefas:

1. Escolha as variáveis numéricas (modelos de Regressão) para serem as variáveis independentes do modelo. 
A variável target é avg_price. Observação: caso julgue necessário, faça a transformação de variáveis categóricas em variáveis numéricas para inputar no modelo. Indique quais variáveis foram transformadas e como foram transformadas
2. Crie partições contendo 75% dos dados para treino e 25% para teste
3. Treine modelos RandomForest (biblioteca RandomForestRegressor) e XGBoost (biblioteca XGBRegressor) para predição dos preços dos carros. Observação: caso julgue necessário, mude os parâmetros dos modelos e rode novos modelos. Indique quais parâmetros foram inputados e indique o treinamento de cada modelo
4. Grave os valores preditos em variáveis criadas
5. Realize a análise de importância das variáveis para estimar a variável target, para cada modelo treinado
6. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados encontrados na análise de importância de variáveis
7. Escolha o melhor modelo com base nas métricas de avaliação MSE, MAE e R²
8. Dê uma breve explicação (máximo de quatro linhas) sobre qual modelo gerou o melhor resultado e a métrica de avaliação utilizada

---
1. Escolha as variáveis numéricas (modelos de Regressão) para serem as variáveis independentes do modelo. 
A variável target é avg_price. Observação: caso julgue necessário, faça a transformação de variáveis categóricas em variáveis numéricas para inputar no modelo. Indique quais variáveis foram transformadas e como foram transformadas

In [ ]:
# Relacionar as colunas da base de dados
colunas_numericas = df.select_dtypes(include="number").columns.tolist()

colunas_categoricas = df.select_dtypes(exclude="number").columns.tolist()

print("Lista de colunas numéricas ", colunas_numericas)
print("\nLista de colunas categóricas ", colunas_categoricas)

In [ ]:
df.dtypes

In [ ]:
# 1. engine_size: Troca a vírgula para ponto e transforma para tipo float
engine_size_num = df['engine_size'].astype(str).str.replace(',', '.').str.strip()
engine_size_num = pd.to_numeric(engine_size_num, errors='coerce')

# 2. gear: Fazer label encoding da variável gear
gear_num = df['gear'].map({'manual': 0, 'automatic': 1})

# 3. fuel: One-Hot Encoding
fuel_dummies = pd.get_dummies(df['fuel'], prefix='fuel', dtype=int, drop_first=False)

# 4. brand: Label Encoding de marcas para código númerico
brand_encoded = pd.DataFrame({'brand_encoded':df['brand'].astype('category').cat.codes})

# Variáveis independentes
features = ['year_of_reference', 'year_model', 'engine_size',
            'gear', 'brand_encoded',
            'fuel_Gasoline', 'fuel_Alcohol', 'fuel_Diesel']

target = 'avg_price_brl'

# Remove linhas com NaN nas features
df_ml = pd.concat([df[colunas_numericas], engine_size_num, gear_num, fuel_dummies, brand_encoded],axis=1)
df_ml = df_ml.dropna()

X_all = df_ml[features]
X_original = df_ml[colunas_numericas].drop([target], axis=1)
y = df_ml[target]

print('=== Variáveis e Transformações ===')
print('\nVariáveis originais transformadas:')
print('  - engine_size  : string com vírgula -> float (ex: "1,6" => 1.6)')
print('  - gear         : Label Encoding -> manual=0, automatic=1')
print('  - fuel         : One-Hot Encoding -> colunas fuel_Gasoline, fuel_Alcohol, fuel_Diesel')
print('  - brand        : Label Encoding -> código inteiro por categoria')
print(f'\nVariáveis independentes (features): {features}')
print(f'Variável target: {target}')
print(f'\nTotal de amostras para modelagem: {len(df_ml):,}')
display(X_all.head())

---
2. Crie partições contendo 75% dos dados para treino e 25% para teste

In [ ]:
# Hyperparametros
rand_state = 13 # para reprodutibilidade dos testes e homenagem ao Grupo!
train_rate = 0.75 # 75% para o treino

In [ ]:
# Cria bases de treino e testes
X_train, X_test, y_train, y_test = train_test_split(X_all, y, train_size=train_rate, random_state=rand_state)


In [ ]:
# Verificação
a = len(X_all)
b = len(X_train)
c = len(X_test)
print(f"Tamanho total do dataset é {a} amostras")
print(f"Numero de amostras para treino : {b} ({b/a:.1%})" )
print(f"Numero de amostras para teste : {c} ({c/a:.1%})" )

---
3. Treine modelos RandomForest (biblioteca RandomForestRegressor) e XGBoost (biblioteca XGBRegressor) para predição dos preços dos carros. Observação: caso julgue necessário, mude os parâmetros dos modelos e rode novos modelos. Indique quais parâmetros foram inputados e indique o treinamento de cada modelo

In [ ]:
# cria preditores RadomForest, XGBRegressor, GradientBoosting
models = {

    "RandomForest": RandomForestRegressor(max_depth=20, 
                                     min_samples_leaf=5, 
                                     min_samples_split=8,
                                     n_estimators=280,
                                     random_state=rand_state,
                                     n_jobs=-1),
    "XGBRegressor": XGBRegressor(),
    "GradientBoosting": GradientBoostingRegressor(),
}


print('=== Modelo 1: RadomForest ===')
print('=== Modelo 2: XGBRegressor ===')
print('=== Modelo 3: GradientBoosting ===')
print('Todos serão treinados com a base com variáveis convertidas para numérica')
print('\n')




---
4. Grave os valores preditos em variáveis criadas

In [ ]:
trained_models = {}

for name, model in models.items():
    start_time = time.time() # inicia contagem de tempo
    
    model.fit(X_train, y_train) # treina
    
    y_hat = model.predict(X_test) # faz a predição
    
    train_time = time.time() - start_time # calcula o tempo do proecesso
    
    trained_models[name] = {
        "model": model,
        "predict": y_hat,
        "train_time": train_time
    } # salva no dicionario
    
    print(f'Tempo total de treinamento e predição :{name} {train_time:.2f} segundos')

---
5. Realize a análise de importância das variáveis para estimar a variável target, para cada modelo treinado

In [ ]:
# Calcula a importancia
# Na minha maquina (i7) levou 2min22
# No Google Colab (CPU) levou quase 6 min
# No Google Colab (GPU) levou 4min24

importance_results = {}

for name, info in trained_models.items():
    model = info["model"]
    result = permutation_importance(
        model,
        X_test,
        y_test,
        n_repeats=20,
        random_state=42,
        n_jobs=-1
    )

    importance_results[name] = result.importances_mean

In [ ]:
# Cria Dataframe comparativo
importance_df = pd.DataFrame(
    importance_results,
    index=X_all.columns
)

importance_df

In [ ]:
# Heatmap comparativo
plt.figure(figsize=(10,6))

sns.heatmap(
    importance_df,
    #cmap="viridis",
    #cmap="rocket_r",
    cmap="magma_r",
    annot=True,
    fmt='.2f'
)

plt.title("Comparação da importância das Features")
plt.show()

In [ ]:
# Ranck médio de importância
importance_df["mean_importance"] = importance_df.mean(axis=1)

importance_df.sort_values(
    "mean_importance",
    ascending=True
)["mean_importance"].plot.barh()

plt.title("Média de importância das Features")
plt.show()

---
6. Dê uma breve explicação (máximo de quatro linhas) sobre os resultados encontrados na análise de importância de variáveis

Nos três modelos testados, **`engine_size`** e **`year_model`** (capacidade em litros do motor e ano do modelo) se destacam como as variáveis mais importante para prever o preço, o que é esperado, pois veículos mais potentes mais novos tendem a ter maior valor de mercado. A **`brand_encoded`** (marca) também aparece com relevância expressiva, refletindo as diferenças de posicionamento e precificação entre as montadoras. As variáveis de combustível contribuem de forma moderada, indicando que o tipo de combustível influencia o preço, mas de forma secundária ao ano e à marca. Já o `year_of_reference` tem importância baixa, sugerindo que a variação de preço entre os anos de referência (2021–2023) é relativamente pequena quando controlada pelas demais características.

---
7. Escolha o melhor modelo com base nas métricas de avaliação MSE, MAE e R²

In [ ]:
# obter o MSE, MAE e R² para cada modelo
def metricas(y_true, y_hat, tempo):
    """Recebe os valores reais, predito e o tempo do modelo e retorna um dicionário com
    o MSE, MAE,  R² e tempo de duração"""
    
    t = round(tempo,2)
    mse_calc = round(mean_squared_error(y_true, y_hat),2)
    mae_calc = round(mean_absolute_error(y_true, y_hat),2)
    r2_calc = round(r2_score(y_true, y_hat) * 100,2)
    metrica = {"mse": mse_calc, "mae": mae_calc, "r2": r2_calc, 'tempo': t}
    return metrica

In [ ]:
# cria um dataframe para guardar as medidas
df_met = pd.DataFrame(columns=['mse','mae','r2', 'tempo'])

# insere no dataframe as metricas calculadas de cada modelo
for name, info in trained_models.items():
    y_hat = info['predict']
    t = info['train_time']
    df_met.loc[name] = metricas(y_test,y_hat, t)
  
print(df_met)

Os melhores modelos com base nas métricas acima foram o *RandomForest* e *XGBRegressor*, tendo a diferença mínima, porém o *XGBRegressor* foi mais rápido no treino e predição.

---
8. Dê uma breve explicação (máximo de quatro linhas) sobre qual modelo gerou o melhor resultado e a métrica de avaliação utilizada

O **XGBRegressor** apresentou um valor de R² muito próximo do **RandomForest**, porém foi mais rápido no tempo de treino e predição, indicando que suas predições se aproximam mais dos preços reais po´rem de forma mais rápida. A métrica **R²** e o tempo foram decisivas na escolha. Embora ambos os modelos tenham apresentado bom desempenho, o **XGBRegressor** se beneficiou de seu mecanismo de boosting, que corrige erros iterativamente e captura padrões mais complexos nos dados. 

### Comparitivo com a base de dados originais

Visando mostrar a importância de entender os dados do problema, fizemos a simulação com os dados numéricos originais

In [ ]:
# usa apenas os valores numéricos originais
print(f'\nVariáveis independentes (features): {colunas_numericas}')
print(f'Variável target: {target}')

X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(X_original, y, train_size=train_rate, random_state=rand_state)

In [ ]:
trained_models_orig = {}

for name, model in models.items():
    start_time = time.time() # inicia contagem de tempo
    
    model.fit(X_train_o, y_train_o) # treina
    
    y_hat = model.predict(X_test_o) # faz a predição
    
    train_time = time.time() - start_time # calcula o tempo do proecesso
    
    trained_models_orig[name] = {
        "model": model,
        "predict": y_hat,
        "train_time": train_time
    } # salva no dicionario
    
    print(f'Tempo total de treinamento e predição :{name} {train_time:.2f} segundos')

In [ ]:
# cria um dataframe para guardar as medidas
df_met_orig = pd.DataFrame(columns=['mse','mae','r2', 'tempo'])

# insere no dataframe as metricas calculadas de cada modelo
for name, info in trained_models_orig.items():
    y_hat = info['predict']
    t = info['train_time']
    df_met_orig.loc[name] = metricas(y_test_o, y_hat, t)
  
print(df_met_orig)

In [ ]:
import matplotlib.ticker as mticker

y_hat = trained_models["XGBRegressor"]["predict"]
y_hat_o = trained_models_orig["XGBRegressor"]["predict"]

fig, ax = plt.subplots(figsize=(10,6))

sample = min(2000, len(y_test))
idx = np.random.choice(len(y_test), sample, replace=False)

ax.scatter(y_test_o.values[idx], y_hat_o[idx], alpha=0.7, color='b', label='Base Original')
ax.scatter(y_test.values[idx], y_hat[idx], alpha=0.7, color='g', label='Base Completa')
ax.plot(
    [y_test_o.min(), y_test_o.max()],
    [y_test_o.min(), y_test_o.max()], color='k')

ax.set_xlabel("Valor Real")
ax.set_ylabel("Valor Predito")
ax.set_title("Modelo XGBRegressor")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))

fig.legend()

fig.suptitle("Comparativo do treino/predição entre os Datasets")

plt.show()